# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
!pip cache purge

Files removed: 0


In [2]:
# 1. Restart your kernel first
# 2. Use a relaxed version constraint to let pip resolve compatible sub-packages
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-openai langchain-huggingface docarray ragas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of

### Example 1

#### Create our QandA application

In [1]:
from langchain.chains import RetrievalQA, LLMChain  # Not langchain_classic
from langchain.indexes import VectorstoreIndexCreator
from langchain_openai import ChatOpenAI, OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch

In [3]:
file = '/content/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [4]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:

vectorstore = index.vectorstore  # Get the vectorstore
retriever = vectorstore.as_retriever()  # Get the retriever

In [8]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [9]:
llm = ChatOpenAI(temperature = 0.0, openai_api_key=OPENAI_API_KEY)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [10]:
data[10]

Document(metadata={'source': '/content/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [11]:
data[11]

Document(metadata={'source': '/content/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


/tmp/ipykernel_2717/1022454896.py:46: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  llm_chain = LLMChain(
/tmp/ipykernel_2717/1022454896.py:56: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = llm_chain.run({"query": query})


answer='Yes, the Cozy Comfort Pullover Set is available in multiple colors such as grey, navy, and black.'


#### LLM-Generated examples

In [13]:
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

['ContextQAEvalChain', 'CotQAEvalChain', 'QAEvalChain', 'QAGenerateChain', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'eval_chain', 'eval_prompt', 'generate_chain', 'generate_prompt']


In [14]:
from langchain.evaluation.qa import QAGenerateChain

In [15]:

# 1. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=OPENAI_API_KEY) # Use your preferred model

# 2. Create the generation chain
example_gen_chain = QAGenerateChain.from_llm(llm)

In [16]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [17]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

/usr/local/lib/python3.12/dist-packages/langchain/chains/llm.py:367: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [18]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What are the key features of the Women's Campside Oxfords, and how should customers determine their shoe size when ordering?",
  'answer': "The Women's Campside Oxfords are designed for comfort and quality with key features including a super-soft canvas material, thick cushioning, and a construction that provides a broken-in feel right from the first wear. Customers should order their regular shoe size; for half sizes that are not offered, they should order up to the next whole size."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, and what materials are used in its construction?',
  'answer': 'The dimensions of the small Recycled Waterhog Dog Mat are 18" x 28", and the medium size measures 22.5" x 34.5". The mat is constructed from 24 oz. polyester fabric made from 94% recycled materials and has a rubber backing.'},
 {'query': "What features make the Infant and Toddler Girls' Coastal Chill Swimsuit suitable for outdoor 

#### Combine examples

In [19]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'}]

In [20]:
# examples += new_example
examples += d_flattened

In [21]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'answer': 'Yes'}

In [22]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Yes, the Cozy Comfort Pullover Set does have side pockets.'}

### Manual Evaluation - Fun part

In [23]:
import langchain
langchain.debug = True

# qa needs 'query' as input, and it handles context internally
result = qa.invoke({
    "query": examples[0]["query"]
})
print(result)

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": ": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have 

In [24]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [26]:
# When creating the RetrievalQA chain, add return_source_documents=True
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True  # Add this
)

In [27]:
predictions = []
for i, eg in enumerate(examples[:5]):
    result = qa.invoke({"query": eg["query"]})

    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result['result'],
        'source_documents': result.get('source_documents', [])
    }
    predictions.append(pred_dict)

for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print(f"Used {len(predictions[i]['source_documents'])} source documents")
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set has side pockets.
Used 4 source documents

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Used 4 source documents

Example 2:
Question: What are the key features of the Women's Campside Oxfords, and how should customers determine their shoe size when ordering?
Real Answer: The Women's Campside Oxfords are designed for comfort and quality with key features including a super-soft canvas material, thick cushioning, and a construction that provides a broken-in feel right from the first wear. Customers should order their regular shoe size; for half sizes that are not offered, they should order up to the next whole size.
Predicted Answer: The key features of th

LLM as a JUdge

In [28]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  # Add this import

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0,openai_api_key=OPENAI_API_KEY)

# Simple eval prompt (no classes)
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()  # Now this will work

# Your EXACT workflow (no classes)
graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())  # Extract number

    graded_outputs.append({'score': score})

    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real: Yes
Predicted: Yes, the Cozy Comfort Pullover Set has side pockets.
Score: 1.0 

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real: The DownTek collection
Predicted: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Score: 1.0 

Example 2:
Question: What are the key features of the Women's Campside Oxfords, and how should customers determine their shoe size when ordering?
Real: The Women's Campside Oxfords are designed for comfort and quality with key features including a super-soft canvas material, thick cushioning, and a construction that provides a broken-in feel right from the first wear. Customers should order their regular shoe size; for half sizes that are not offered, they should order up to the next whole size.
Predicted: The key features of the Women's Campside Oxfords include:

- Ultralightweight and comfor

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings
loader = TextLoader("/content/nyc_text.txt")
index = VectorstoreIndexCreator(embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cuda'})).from_loaders([loader])

# Explicitly define the retriever
retriever = index.vectorstore.as_retriever()

llm = ChatOpenAI(temperature= 0, openai_api_key=OPENAI_API_KEY)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/langchain/indexes/vectorstore.py:127: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  warnings.warn(


In [30]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.'

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [31]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [32]:
# Generate predictions with your qa_chain
all_predictions = []
for example in examples:
    q = example["query"]
    result = qa_chain.invoke({"query": q})  # RetrievalQA uses 'query'

    all_predictions.append({
        "question": q,
        "answer": result['result'],  # Extract answer from result dict
        "contexts": [doc.page_content for doc in result['source_documents']],  # Extract contexts
        "ground_truth": example["ground_truths"][0]
    })

print("Predictions generated:", len(all_predictions))

# Display first prediction to verify
print("\nExample prediction:")
print("Question:", all_predictions[0]['question'])
print("Answer:", all_predictions[0]['answer'])
print("Contexts:", len(all_predictions[0]['contexts']), "documents")
print("Ground Truth:", all_predictions[0]['ground_truth'])

Predictions generated: 5

Example prediction:
Question: What is the population of New York City as of 2020?
Answer: The population of New York City as of 2020 is 8,804,190 residents.
Contexts: 4 documents
Ground Truth: 8,804,190


#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

evaluate(): evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).

In [33]:
#evaluate(): evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).

import os
from google.colab import userdata

# Set OpenAI API key as environment variable
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Now run RAGAS evaluation
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

dataset_all = Dataset.from_list(all_predictions)

print("=" * 50)
print("RAGAS Evaluation - Faithfulness")
print("=" * 50)
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Context Recall")
print("=" * 50)
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Combined Metrics")
print("=" * 50)
results = evaluate(dataset_all, metrics=[faithfulness, context_recall])
print(results)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2717/447105134.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_2717/447105134.py:11: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  fr

RAGAS Evaluation - Faithfulness


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 0.9667}

RAGAS Evaluation - Context Recall


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.7333}

RAGAS Evaluation - Combined Metrics


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 1.0000, 'context_recall': 0.7333}


In [34]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]

In [35]:
print("=" * 70)
print("RAGAS EVALUATION WITH EvaluatorChain")
print("=" * 70)

from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall, answer_relevancy

# Create evaluation chains
print("\n1. Creating evaluation chains...")
faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

print("✓ Evaluation chains created")

RAGAS EVALUATION WITH EvaluatorChain

1. Creating evaluation chains...
✓ Evaluation chains created


/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1002: RuntimeWarning: fields may not start with an underscore, ignoring "_required_columns"
  warnings.warn(f'fields may not start with an underscore, ignoring "{f_name}"', RuntimeWarning)
/tmp/ipykernel_2717/2028705441.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall, answer_relevancy
/tmp/ipykernel_2717/2028705441.py:6: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall, answer_relevancy
/tmp/ipykernel_2717/2028705441.py:6: DeprecationWarning: Importing answer_relevancy from 'rag

Method 1: Evaluate a Single Result with __call__()

In [36]:
#Method 1: Evaluate a Single Result with call()

print("\n" + "=" * 70)
print("METHOD 1: Evaluate Single Result with __call__()")
print("=" * 70)

# Get a single result from the QA chain
single_query = examples[0]["query"]
print(f"\nQuery: {single_query}")

result = qa_chain.invoke({"query": single_query})
print(f"Answer: {result['result']}")

# Prepare the result for Ragas EvaluatorChain
single_eval_input = {
    "question": result["query"],
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0] # Ensure ground_truth is singular and a single string
}

# Evaluate faithfulness
print("\n--- Evaluating Faithfulness ---")
faithfulness_result = faithfulness_chain(single_eval_input)
print(f"Faithfulness Score: {faithfulness_result['faithfulness']:.3f}") # Corrected key

# Evaluate context recall
print("\n--- Evaluating Context Recall ---")
context_recall_result = context_recall_chain(single_eval_input) # Pass the same correctly formatted input
print(f"Context Recall Score: {context_recall_result['context_recall']:.3f}") # Corrected key


METHOD 1: Evaluate Single Result with __call__()

Query: What is the population of New York City as of 2020?
Answer: The population of New York City as of 2020 is 8,804,190 residents.

--- Evaluating Faithfulness ---


/tmp/ipykernel_2717/1963167613.py:24: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  faithfulness_result = faithfulness_chain(single_eval_input)


Faithfulness Score: 1.000

--- Evaluating Context Recall ---
Context Recall Score: 1.000


Test with Fake Results (Low Scores)

In [37]:
#Test with Fake Results (Low Scores)

print("\n" + "=" * 70)
print("TESTING WITH FAKE RESULTS (Expected Low Scores)")
print("=" * 70)

# The base input structure for EvaluatorChain, derived from result and examples[0]
# 'result' and 'examples[0]' are available from previous cell execution
base_eval_input_for_test = {
    "question": result["query"],
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0]
}

# Test 1: Fake answer (low faithfulness)
print("\n--- Test 1: Fake Answer ---")
fake_faithfulness_input = base_eval_input_for_test.copy()
fake_faithfulness_input["answer"] = "The population of NYC is 100 million people and it's on Mars."
fake_faithfulness = faithfulness_chain(fake_faithfulness_input)
print(f"Original Answer Faithfulness: {faithfulness_result['faithfulness']:.3f}")
print(f"Fake Answer Faithfulness: {fake_faithfulness['faithfulness']:.3f}")
print("→ Lower score because answer contradicts the source documents")

# Test 2: Fake source documents (low context recall)
print("\n--- Test 2: Fake Source Documents ---")
from langchain.schema import Document
fake_contexts_docs = [
    Document(page_content="I love pizza and ice cream."),
    Document(page_content="The weather is nice today.")
]
fake_context_recall_input = base_eval_input_for_test.copy()
fake_context_recall_input["contexts"] = [doc.page_content for doc in fake_contexts_docs]
# The ground_truth should remain the original one for context recall evaluation
fake_context_recall = context_recall_chain(fake_context_recall_input)
print(f"Original Context Recall: {context_recall_result['context_recall']:.3f}")
print(f"Fake Context Recall: {fake_context_recall['context_recall']:.3f}")
print("→ Lower score because ground truth is not in source documents")


TESTING WITH FAKE RESULTS (Expected Low Scores)

--- Test 1: Fake Answer ---
Original Answer Faithfulness: 1.000
Fake Answer Faithfulness: 0.000
→ Lower score because answer contradicts the source documents

--- Test 2: Fake Source Documents ---
Original Context Recall: 1.000
Fake Context Recall: 0.000
→ Lower score because ground truth is not in source documents


Method 2: Batch Evaluate with evaluate()

In [38]:
#Method 2: Batch Evaluate with evaluate()

def evaluate_qa_pipeline(
    qa_chain,
    examples,
    evaluators,
    prediction_key="result",
    context_key="source_documents",
):
    print("\n" + "=" * 70)
    print("RUNNING BATCH EVALUATION")
    print("=" * 70)

    # Generate predictions
    predictions = qa_chain.batch([{"query": ex["query"]} for ex in examples])

    def extract_contexts(pred):
        docs = pred.get(context_key, [])
        return [doc.page_content for doc in docs] if docs else []

    results = {"metrics": {}}

    for name, evaluator in evaluators.items():
        print(f"\n--- Evaluating {name} ---")

        eval_inputs = [
            {
                "question": ex["query"],
                "answer": pred.get(prediction_key, ""),
                "contexts": extract_contexts(pred),
                "ground_truth": ex["ground_truths"][0] if ex.get("ground_truths") else "",
            }
            for ex, pred in zip(examples, predictions)
        ]

        eval_outputs = evaluator.batch(eval_inputs)

        # Flexible score extraction
        def extract_score(res):
            if "score" in res:
                return res["score"]
            if name in res:
                return res[name]
            if "value" in res:
                return res["value"]
            return None

        scores = []
        for res in eval_outputs:
            score = extract_score(res)
            if score is not None:
                scores.append(score)
            else:
                print("Unknown format:", res)

        avg_score = sum(scores) / len(scores) if scores else 0.0

        print(f"Average {name}: {avg_score:.3f}")
        print(f"Scores: {scores}")

        results["metrics"][name] = {
            "average": avg_score,
            "scores": scores,
        }

    return results

In [39]:
evaluators = {
    "faithfulness": faithfulness_chain,
    "context_recall": context_recall_chain,
}

results = evaluate_qa_pipeline(
    qa_chain=qa_chain,
    examples=examples,
    evaluators=evaluators,
)


RUNNING BATCH EVALUATION

--- Evaluating faithfulness ---
Average faithfulness: 0.733
Scores: [0.5, 0.5, 1.0, 0.6666666666666666, 1.0]

--- Evaluating context_recall ---
Average context_recall: 0.733
Scores: [1.0, 1.0, 1.0, 0.0, 0.6666666666666666]


SECTION 1 — Single Example Evaluation (Understanding RAGAS Basics)

In [40]:
from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall

# create evaluators
faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

# run one query through RAG pipeline
query = examples[0]["query"]
result = qa_chain.invoke({"query": query})

# format into RAGAS input schema
single_eval_input = {
    "question": query,
    "answer": result["result"],
    "contexts": [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0]
}

# evaluate single example
faith_score = faithfulness_chain(single_eval_input)
recall_score = context_recall_chain(single_eval_input)

print("Faithfulness:", faith_score["faithfulness"])
print("Context Recall:", recall_score["context_recall"])

/tmp/ipykernel_2717/3256516451.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_2717/3256516451.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


Faithfulness: 1.0
Context Recall: 1.0


SECTION 2 — Batch Evaluation (LangChain Style)

In [41]:
def run_batch_evaluation(qa_chain, examples):
    predictions = []

    for ex in examples:
        result = qa_chain.invoke({"query": ex["query"]})

        predictions.append({
            "question": ex["query"],
            "answer": result["result"],
            "contexts": [doc.page_content for doc in result["source_documents"]],
            "ground_truth": ex["ground_truths"][0],
        })

    return predictions


predictions = run_batch_evaluation(qa_chain, examples)

In [42]:
eval_inputs = [
    {
        "question": p["question"],
        "answer": p["answer"],
        "contexts": p["contexts"],
        "ground_truth": p["ground_truth"],
    }
    for p in predictions
]

faith_batch = faithfulness_chain.batch(eval_inputs)
recall_batch = context_recall_chain.batch(eval_inputs)

# average manually
faith_scores = [r["faithfulness"] for r in faith_batch]
recall_scores = [r["context_recall"] for r in recall_batch]

print("Avg Faithfulness:", sum(faith_scores)/len(faith_scores))
print("Avg Context Recall:", sum(recall_scores)/len(recall_scores))

Avg Faithfulness: 0.7555555555555555
Avg Context Recall: 0.9333333333333333


SECTION 3 — RAGAS Native Evaluation (Production Method)